# DualPath-AttentionNet - Fashion-MNIST Classifier

---

## DESAFIO 03e - AC3

**Objetivo:** Criar uma arquitetura CNN inédita para classificar imagens do Fashion-MNIST

**Dataset:** Fashion-MNIST (60.000 treino + 10.000 teste)

**Classes:** 10 categorias de roupas e acessórios

**Métrica:** Acurácia no conjunto de teste (4 casas decimais)

---

# ARQUITETURA: DualPath-AttentionNet

## Conceito Inovador

A **DualPath-AttentionNet** é uma arquitetura completamente inédita que combina:

### 1. **Processamento Dual-Path (Dois Caminhos Paralelos)**
- **Path 1 (Detail):** Captura detalhes finos com filtros pequenos (3x3)
- **Path 2 (Context):** Captura contexto amplo com filtros maiores (5x5) e convoluções dilatadas

### 2. **Mecanismo de Atenção Espacial**
- Aprende automaticamente quais regiões da imagem são mais importantes
- Implementação customizada inspirada em CBAM (Woo et al., 2018)

### 3. **Fusão Adaptativa**
- Os dois caminhos são fundidos através de convolução 1x1
- Aprende a combinar informações de diferentes escalas

### 4. **Convoluções Dilatadas**
- Aumentam o receptive field sem aumentar parâmetros
- Baseado em Yu & Koltun, 2016

### 5. **Regularização Avançada**
- Batch Normalization (Ioffe & Szegedy, 2015)
- Spatial Dropout (Tompson et al., 2015)
- L2 Regularization

---

## Por que esta arquitetura é inédita?

**Diferenças das arquiteturas clássicas:**

| Arquitetura | Características | Nossa diferença |
|------------|-----------------|------------------|
| LeNet | 2 conv layers, simples | Dual-path + attention |
| AlexNet | 5 conv layers, ReLU | Multi-escala paralela |
| VGG | Múltiplas 3x3 sequenciais | Processamento paralelo |
| ResNet | Residual connections | Dual-path com fusão |
| InceptionNet | Inception modules | Atenção espacial integrada |

**Nosso diferencial:**
- Combinação única de dual-path + atenção espacial + fusão adaptativa
- Não replica nenhuma arquitetura existente
- Otimizada especificamente para Fashion-MNIST

---

# Referências Teóricas

## Principais Trabalhos que Inspiraram (sem replicar):

1. **Feature Pyramid Networks (Lin et al., 2017)**
   - Conceito: Multi-scale feature processing
   - Nossa adaptação: Dual-path paralelo ao invés de pirâmide sequencial

2. **CBAM: Convolutional Block Attention Module (Woo et al., 2018)**
   - Conceito: Attention mechanisms em CNNs
   - Nossa adaptação: Atenção espacial customizada para imagens 28x28

3. **Multi-Scale Context Aggregation (Yu & Koltun, 2016)**
   - Conceito: Dilated convolutions
   - Nossa adaptação: Integração no path de contexto

4. **Network in Network (Lin et al., 2014)**
   - Conceito: 1x1 convolutions
   - Nossa adaptação: Fusão adaptativa dos paths

5. **Batch Normalization (Ioffe & Szegedy, 2015)**
   - Estabilização do treinamento
   - Aplicado em todos os blocos

6. **Dropout (Srivastava et al., 2014)**
   - Regularização
   - Usamos spatial dropout para preservar coerência espacial

---

# Importações e Configurações

In [1]:
# Instalação de dependências (se necessário no Colab)
!pip install -q tensorflow matplotlib seaborn scikit-learn numpy


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import json
from datetime import datetime

# Configurar para resultados reproduzíveis
np.random.seed(42)
tf.random.set_seed(42)

# Configurar estilo de plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponível: {tf.config.list_physical_devices('GPU')}")

TensorFlow version: 2.20.0
GPU disponível: []


# ⚙️ Configurações e Hiperparâmetros

## Justificativas das Escolhas:

### Arquitetura:
- **IMG_SIZE:** 28x28 (padrão do Fashion-MNIST)
- **NUM_CLASSES:** 10 classes

### Treinamento:
- **BATCH_SIZE:** 64 - Balanceamento entre estabilidade (batch maior) e velocidade
- **EPOCHS:** 100 - Com early stopping, para não subtratar
- **LEARNING_RATE:** 0.001 - Taxa padrão do Adam, empiricamente eficaz

### Regularização:
- **L2_REG:** 1e-4 - Previne pesos muito grandes
- **DROPOUT_RATE:** 0.3 - Dropout moderado (30%)
- **SPATIAL_DROPOUT:** 0.2 - Preserva estrutura espacial

### Data Augmentation:
- **ROTATION:** ±10° - Roupas podem estar em pequenos ângulos
- **SHIFT:** ±10% - Objetos podem não estar perfeitamente centralizados
- **ZOOM:** ±10% - Simula diferentes distâncias
- **Sem flip horizontal:** Orientação importa (sapatos, bolsas)

In [ ]:
class Config:
    """Configurações do modelo"""
    # Dados
    IMG_SIZE = 28
    IMG_CHANNELS = 1
    NUM_CLASSES = 10
    
    # Treinamento
    BATCH_SIZE = 64
    EPOCHS = 100
    LEARNING_RATE = 0.001
    
    # Regularização
    L2_REG = 1e-4
    DROPOUT_RATE = 0.3
    SPATIAL_DROPOUT = 0.2
    
    # Data augmentation
    ROTATION_RANGE = 10
    WIDTH_SHIFT = 0.1
    HEIGHT_SHIFT = 0.1
    ZOOM_RANGE = 0.1
    
    # Classes
    CLASS_NAMES = [
        'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
        'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
    ]

print("✓ Configurações carregadas")

# 📊 1. Análise Exploratória dos Dados

In [ ]:
# Carregar dados
print("Carregando Fashion-MNIST...")
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()

print(f"\n📦 Dataset Shape:")
print(f"  Treino: {X_train_full.shape}")
print(f"  Teste: {X_test.shape}")
print(f"\n📊 Range de pixels: [{X_train_full.min()}, {X_train_full.max()}]")
print(f"📊 Classes: {np.unique(y_train_full)}")

In [ ]:
# Visualizar distribuição de classes
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Treino
unique, counts = np.unique(y_train_full, return_counts=True)
axes[0].bar(unique, counts, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Classe', fontsize=12)
axes[0].set_ylabel('Quantidade', fontsize=12)
axes[0].set_title('Distribuição de Classes - Treino', fontsize=14, fontweight='bold')
axes[0].set_xticks(unique)
axes[0].set_xticklabels(Config.CLASS_NAMES, rotation=45, ha='right')
axes[0].grid(axis='y', alpha=0.3)

# Teste
unique, counts = np.unique(y_test, return_counts=True)
axes[1].bar(unique, counts, color='coral', edgecolor='black')
axes[1].set_xlabel('Classe', fontsize=12)
axes[1].set_ylabel('Quantidade', fontsize=12)
axes[1].set_title('Distribuição de Classes - Teste', fontsize=14, fontweight='bold')
axes[1].set_xticks(unique)
axes[1].set_xticklabels(Config.CLASS_NAMES, rotation=45, ha='right')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Distribuição balanceada: ~6000 por classe no treino, ~1000 no teste")

In [ ]:
# Visualizar exemplos de cada classe
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.ravel()

for i in range(10):
    # Pegar primeira imagem de cada classe
    idx = np.where(y_train_full == i)[0][0]
    axes[i].imshow(X_train_full[idx], cmap='gray')
    axes[i].set_title(f'{i}: {Config.CLASS_NAMES[i]}', fontsize=10, fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Exemplo de Cada Classe - Fashion-MNIST', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 🔍 Observações da Análise Exploratória:

1. **Dataset balanceado:** ~6000 samples por classe (treino), ~1000 por classe (teste)
2. **Imagens pequenas:** 28x28 pixels em grayscale
3. **Desafios:**
   - Algumas classes são similares (T-shirt vs Shirt, Sneaker vs Ankle boot)
   - Variações dentro da mesma classe (diferentes estilos de roupas)
   - Necessidade de capturar tanto detalhes quanto contexto

**→ Justifica nossa arquitetura dual-path com atenção!**

# 🏗️ 2. Construção da Arquitetura

## 2.1 Camada de Atenção Espacial

### Descrição Matemática:

Dado um feature map $F \in \mathbb{R}^{H \times W \times C}$:

1. **Feature Aggregation:**
   $$f_{avg} = \text{AvgPool}_C(F) \in \mathbb{R}^{H \times W \times 1}$$
   $$f_{max} = \text{MaxPool}_C(F) \in \mathbb{R}^{H \times W \times 1}$$

2. **Concatenação:**
   $$f_{concat} = [f_{avg}; f_{max}] \in \mathbb{R}^{H \times W \times 2}$$

3. **Attention Map:**
   $$A = \sigma(\text{Conv}_{3 \times 3}(f_{concat})) \in \mathbb{R}^{H \times W \times 1}$$
   
   onde $\sigma$ é a função sigmoid

4. **Output:**
   $$F' = F \odot A$$
   
   onde $\odot$ é multiplicação element-wise

### Parâmetros:
- Conv3x3: $(3 \times 3 \times 2 + 1) \times 1 = 19$ parâmetros

### Referência:
Adaptado de: Woo et al., 2018 - "CBAM: Convolutional Block Attention Module"

In [ ]:
class SpatialAttention(layers.Layer):
    """
    Camada de Atenção Espacial Customizada
    
    Aprende a dar peso diferenciado para diferentes regiões espaciais.
    """
    def __init__(self, kernel_size=3, **kwargs):
        super(SpatialAttention, self).__init__(**kwargs)
        self.kernel_size = kernel_size
    
    def build(self, input_shape):
        self.conv = layers.Conv2D(
            filters=1,
            kernel_size=self.kernel_size,
            padding='same',
            activation='sigmoid',
            kernel_initializer='he_normal',
            name='attention_conv'
        )
        super(SpatialAttention, self).build(input_shape)
    
    def call(self, inputs):
        # Average e Max pooling ao longo dos canais
        avg_pool = tf.reduce_mean(inputs, axis=-1, keepdims=True)
        max_pool = tf.reduce_max(inputs, axis=-1, keepdims=True)
        
        # Concatenar
        concat = tf.concat([avg_pool, max_pool], axis=-1)
        
        # Gerar mapa de atenção
        attention_map = self.conv(concat)
        
        # Aplicar atenção
        return inputs * attention_map
    
    def get_config(self):
        config = super(SpatialAttention, self).get_config()
        config.update({'kernel_size': self.kernel_size})
        return config

print("✓ Camada SpatialAttention definida")

## 2.2 Bloco Dual-Path

### Arquitetura do Bloco:

```
Input [H, W, C_in]
  ├─→ Path 1 (Detail):
  │     Conv3x3(filters) → BN → ReLU
  │     → Conv3x3(filters) → BN → ReLU
  │     → [H, W, filters]
  │
  └─→ Path 2 (Context):
        Conv5x5(filters) → BN → ReLU
        → DilatedConv3x3(filters, d=2) → BN → ReLU
        → [H, W, filters]

Concatenate[Path1, Path2] → [H, W, 2*filters]
  → Conv1x1(filters) (Fusion)
  → BN → ReLU
  → SpatialDropout
  → SpatialAttention
  → [H, W, filters]
```

### Cálculo de Parâmetros:

Para um bloco com `filters=F` e entrada `C_in`:

**Path 1:**
- Conv3x3: $(3 \times 3 \times C_{in} + 1) \times F$
- Conv3x3: $(3 \times 3 \times F + 1) \times F$

**Path 2:**
- Conv5x5: $(5 \times 5 \times C_{in} + 1) \times F$
- DilatedConv3x3: $(3 \times 3 \times F + 1) \times F$

**Fusion:**
- Conv1x1: $(2F + 1) \times F$

**Total aproximado:** $O((9C_{in} + 25C_{in} + 18F + 2F) \times F)$

### Justificativas:

1. **Path 1 (3x3 duplo):**
   - Receptive field: 5x5 efetivo
   - Captura detalhes locais, texturas
   - Similar ao VGG, mas em paralelo

2. **Path 2 (5x5 + dilated 3x3):**
   - Receptive field: ~11x11 efetivo
   - Captura contexto global, formas
   - Dilatação aumenta RF sem custos

3. **Fusão 1x1:**
   - Aprende combinação ótima
   - Reduz dimensionalidade (2F → F)
   - Baseado em Network-in-Network

4. **Spatial Dropout:**
   - Regularização espacialmente coerente
   - Melhor que dropout comum em CNNs

5. **Atenção:**
   - Foca em regiões importantes
   - Aprende automaticamente

In [ ]:
class DualPathBlock(layers.Layer):
    """
    Bloco Dual-Path com Atenção
    
    Processa entrada através de dois caminhos paralelos:
    - Path 1: Detalhes (3x3)
    - Path 2: Contexto (5x5 + dilated)
    """
    def __init__(self, filters, dilation_rate=2, **kwargs):
        super(DualPathBlock, self).__init__(**kwargs)
        self.filters = filters
        self.dilation_rate = dilation_rate
        
        # Path 1: Detail
        self.conv1_1 = layers.Conv2D(
            filters, 3, padding='same',
            kernel_regularizer=regularizers.l2(Config.L2_REG),
            kernel_initializer='he_normal',
            name='path1_conv1'
        )
        self.bn1_1 = layers.BatchNormalization(name='path1_bn1')
        self.conv1_2 = layers.Conv2D(
            filters, 3, padding='same',
            kernel_regularizer=regularizers.l2(Config.L2_REG),
            kernel_initializer='he_normal',
            name='path1_conv2'
        )
        self.bn1_2 = layers.BatchNormalization(name='path1_bn2')
        
        # Path 2: Context
        self.conv2_1 = layers.Conv2D(
            filters, 5, padding='same',
            kernel_regularizer=regularizers.l2(Config.L2_REG),
            kernel_initializer='he_normal',
            name='path2_conv1'
        )
        self.bn2_1 = layers.BatchNormalization(name='path2_bn1')
        self.conv2_2 = layers.Conv2D(
            filters, 3, padding='same',
            dilation_rate=dilation_rate,
            kernel_regularizer=regularizers.l2(Config.L2_REG),
            kernel_initializer='he_normal',
            name='path2_conv2_dilated'
        )
        self.bn2_2 = layers.BatchNormalization(name='path2_bn2')
        
        # Fusion
        self.fusion_conv = layers.Conv2D(
            filters, 1, padding='same',
            kernel_regularizer=regularizers.l2(Config.L2_REG),
            kernel_initializer='he_normal',
            name='fusion_conv'
        )
        self.fusion_bn = layers.BatchNormalization(name='fusion_bn')
        
        # Regularização e atenção
        self.spatial_dropout = layers.SpatialDropout2D(
            Config.SPATIAL_DROPOUT, name='spatial_dropout'
        )
        self.attention = SpatialAttention(kernel_size=3)
        self.relu = layers.ReLU()
    
    def call(self, inputs, training=None):
        # Path 1: Detail processing
        x1 = self.conv1_1(inputs)
        x1 = self.bn1_1(x1, training=training)
        x1 = self.relu(x1)
        x1 = self.conv1_2(x1)
        x1 = self.bn1_2(x1, training=training)
        x1 = self.relu(x1)
        
        # Path 2: Context processing
        x2 = self.conv2_1(inputs)
        x2 = self.bn2_1(x2, training=training)
        x2 = self.relu(x2)
        x2 = self.conv2_2(x2)
        x2 = self.bn2_2(x2, training=training)
        x2 = self.relu(x2)
        
        # Concatenate and fuse
        x = tf.concat([x1, x2], axis=-1)
        x = self.fusion_conv(x)
        x = self.fusion_bn(x, training=training)
        x = self.relu(x)
        
        # Regularization and attention
        x = self.spatial_dropout(x, training=training)
        x = self.attention(x)
        
        return x
    
    def get_config(self):
        config = super(DualPathBlock, self).get_config()
        config.update({
            'filters': self.filters,
            'dilation_rate': self.dilation_rate
        })
        return config

print("✓ Bloco DualPathBlock definido")

## 2.3 Modelo Completo: DualPath-AttentionNet

### Arquitetura Completa:

```
Input (28, 28, 1)
  ↓
┌─────────────────────────────────────┐
│ BLOCO INICIAL: Feature Extraction  │
│ Conv3x3(32) → BN → ReLU → Dropout  │
│ Output: (28, 28, 32)                │
└─────────────────────────────────────┘
  ↓
┌─────────────────────────────────────┐
│ BLOCO 1: DualPath(64)              │
│ - Captura features de baixo nível  │
│ Output: (28, 28, 64)                │
└─────────────────────────────────────┘
  ↓ MaxPooling2D
┌─────────────────────────────────────┐
│ (14, 14, 64)                        │
└─────────────────────────────────────┘
  ↓
┌─────────────────────────────────────┐
│ BLOCO 2: DualPath(128)             │
│ - Captura features de médio nível  │
│ Output: (14, 14, 128)               │
└─────────────────────────────────────┘
  ↓ MaxPooling2D
┌─────────────────────────────────────┐
│ (7, 7, 128)                         │
└─────────────────────────────────────┘
  ↓
┌─────────────────────────────────────┐
│ BLOCO 3: DualPath(256, dilation=3) │
│ - Captura features de alto nível   │
│ - SEM pooling (preserva info)      │
│ Output: (7, 7, 256)                 │
└─────────────────────────────────────┘
  ↓
┌─────────────────────────────────────┐
│ GLOBAL POOLING                      │
│ GAP + GMP → Concatenate             │
│ Output: (512,)                      │
└─────────────────────────────────────┘
  ↓
┌─────────────────────────────────────┐
│ CLASSIFICATION HEAD                 │
│ Dense(256) → BN → ReLU → Dropout    │
│ Dense(128) → BN → ReLU → Dropout    │
│ Dense(10, softmax)                  │
└─────────────────────────────────────┘
  ↓
Output (10,) - Probabilidades
```

### Progressão de Features:

| Camada | Shape | RF Efetivo | Descrição |
|--------|-------|------------|------------|
| Input | (28,28,1) | 1x1 | Imagem raw |
| Initial | (28,28,32) | 3x3 | Features básicas |
| Block1 | (28,28,64) | ~11x11 | Edges, texturas |
| Pool1 | (14,14,64) | 22x22 | Downsampling |
| Block2 | (14,14,128) | ~44x44 | Partes, padrões |
| Pool2 | (7,7,128) | 88x88 | Downsampling |
| Block3 | (7,7,256) | ~150x150 | Objetos completos |
| Global | (512,) | Full | Agregação |
| Output | (10,) | - | Classes |

### Estimativa de Parâmetros:

- **Initial Conv:** ~320 params
- **DualPath Block 1:** ~152K params
- **DualPath Block 2:** ~608K params
- **DualPath Block 3:** ~2.4M params
- **Classification:** ~165K params

**Total: ~3.3M parâmetros**

### Justificativas das Escolhas:

1. **Progressão 32→64→128→256:**
   - Aumento gradual de capacidade
   - Features cada vez mais abstratas
   - Padrão comprovado (VGG, ResNet)

2. **MaxPooling após blocos 1 e 2:**
   - Reduz 28→14→7
   - Aumenta receptive field
   - Invariância a translações
   - Não usado em bloco 3 (preserva info)

3. **Global Pooling (GAP + GMP):**
   - GAP: Informação média
   - GMP: Features salientes
   - Combinação rica de informação
   - Não há overfitting espacial

4. **Classification Head (256→128→10):**
   - Redução gradual
   - Dropout para regularização
   - BN para estabilidade

5. **Batch Normalization:**
   - Acelera convergência
   - Permite LR maiores
   - Reduz internal covariate shift

In [ ]:
def create_dualpath_attention_net():
    """
    Cria o modelo DualPath-AttentionNet completo
    """
    inputs = keras.Input(
        shape=(Config.IMG_SIZE, Config.IMG_SIZE, Config.IMG_CHANNELS),
        name='input'
    )
    
    # ===== BLOCO INICIAL =====
    x = layers.Conv2D(
        32, 3, padding='same',
        kernel_regularizer=regularizers.l2(Config.L2_REG),
        kernel_initializer='he_normal',
        name='initial_conv'
    )(inputs)
    x = layers.BatchNormalization(name='initial_bn')(x)
    x = layers.ReLU(name='initial_relu')(x)
    x = layers.SpatialDropout2D(Config.SPATIAL_DROPOUT, name='initial_dropout')(x)
    
    # ===== BLOCO 1 =====
    x = DualPathBlock(64, name='dualpath_block_1')(x)
    x = layers.MaxPooling2D(2, name='pool_1')(x)
    
    # ===== BLOCO 2 =====
    x = DualPathBlock(128, name='dualpath_block_2')(x)
    x = layers.MaxPooling2D(2, name='pool_2')(x)
    
    # ===== BLOCO 3 =====
    x = DualPathBlock(256, dilation_rate=3, name='dualpath_block_3')(x)
    
    # ===== GLOBAL POOLING =====
    gap = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    gmp = layers.GlobalMaxPooling2D(name='global_max_pool')(x)
    x = layers.Concatenate(name='global_concat')([gap, gmp])
    
    # ===== CLASSIFICATION HEAD =====
    x = layers.Dense(
        256,
        kernel_regularizer=regularizers.l2(Config.L2_REG),
        kernel_initializer='he_normal',
        name='dense_1'
    )(x)
    x = layers.BatchNormalization(name='dense_bn_1')(x)
    x = layers.ReLU(name='dense_relu_1')(x)
    x = layers.Dropout(Config.DROPOUT_RATE, name='dropout_1')(x)
    
    x = layers.Dense(
        128,
        kernel_regularizer=regularizers.l2(Config.L2_REG),
        kernel_initializer='he_normal',
        name='dense_2'
    )(x)
    x = layers.BatchNormalization(name='dense_bn_2')(x)
    x = layers.ReLU(name='dense_relu_2')(x)
    x = layers.Dropout(Config.DROPOUT_RATE, name='dropout_2')(x)
    
    outputs = layers.Dense(
        Config.NUM_CLASSES,
        activation='softmax',
        kernel_initializer='glorot_uniform',
        name='output'
    )(x)
    
    # Criar modelo
    model = keras.Model(
        inputs=inputs,
        outputs=outputs,
        name='DualPath_AttentionNet'
    )
    
    return model

print("✓ Função create_dualpath_attention_net() definida")

In [ ]:
# Criar e visualizar o modelo
model = create_dualpath_attention_net()
print("\n" + "="*70)
print(" ARQUITETURA DO MODELO: DUALPATH-ATTENTIONNET")
print("="*70)
model.summary()

# Contar parâmetros
trainable = model.count_params()
print(f"\n📊 Total de parâmetros treináveis: {trainable:,}")

# 📦 3. Preparação dos Dados

## 3.1 Carregamento e Preprocessamento

### Pipeline de Preprocessamento:

1. **Normalização:** $x' = \frac{x}{255}$ ∈ [0, 1]
   - Facilita convergência
   - Estabiliza gradientes

2. **Reshape:** (28, 28) → (28, 28, 1)
   - Adiciona dimensão de canal
   - Necessário para TensorFlow

3. **Split:** 80% treino, 20% validação
   - 48.000 treino
   - 12.000 validação
   - 10.000 teste

4. **One-hot encoding:**
   - 5 → [0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
   - Necessário para categorical crossentropy

In [ ]:
def load_and_preprocess_data():
    """Carrega e preprocessa Fashion-MNIST"""
    print("Carregando Fashion-MNIST...")
    (X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()
    
    # Normalização
    X_train_full = X_train_full.astype('float32') / 255.0
    X_test = X_test.astype('float32') / 255.0
    
    # Reshape
    X_train_full = X_train_full.reshape(-1, Config.IMG_SIZE, Config.IMG_SIZE, 1)
    X_test = X_test.reshape(-1, Config.IMG_SIZE, Config.IMG_SIZE, 1)
    
    # Split treino/validação (80/20)
    num_val = int(0.2 * len(X_train_full))
    X_val = X_train_full[-num_val:]
    y_val = y_train_full[-num_val:]
    X_train = X_train_full[:-num_val]
    y_train = y_train_full[:-num_val]
    
    # One-hot encoding
    y_train = to_categorical(y_train, Config.NUM_CLASSES)
    y_val = to_categorical(y_val, Config.NUM_CLASSES)
    y_test_categorical = to_categorical(y_test, Config.NUM_CLASSES)
    
    print(f"\n✓ Dados processados:")
    print(f"  Treino: {X_train.shape}")
    print(f"  Validação: {X_val.shape}")
    print(f"  Teste: {X_test.shape}")
    
    return (X_train, y_train), (X_val, y_val), (X_test, y_test, y_test_categorical)

# Carregar dados
(X_train, y_train), (X_val, y_val), (X_test, y_test, y_test_cat) = load_and_preprocess_data()

## 3.2 Data Augmentation

### Transformações Aplicadas:

1. **Rotação:** $\theta \in [-10°, +10°]$
2. **Shift Horizontal:** $\Delta x \in [-10\%, +10\%]$
3. **Shift Vertical:** $\Delta y \in [-10\%, +10\%]$
4. **Zoom:** $z \in [0.9, 1.1]$

### Por que essas transformações?

- **Rotação moderada (±10°):** Roupas podem estar em pequenos ângulos
- **Translação (±10%):** Objetos podem não estar perfeitamente centralizados
- **Zoom (±10%):** Simula diferentes distâncias/tamanhos
- **SEM flip horizontal:** Orientação importa (sapatos direito/esquerdo, bolsas)
- **Transformações sutis:** Imagens já são pequenas (28x28)

### Benefícios:

1. Aumenta efetivamente o tamanho do dataset
2. Reduz overfitting
3. Melhora generalização
4. Simula variações naturais

### Referência:
Perez & Wang, 2017 - "The Effectiveness of Data Augmentation in Image Classification"

In [ ]:
def create_data_augmentation():
    """Cria pipeline de data augmentation"""
    return keras.preprocessing.image.ImageDataGenerator(
        rotation_range=Config.ROTATION_RANGE,
        width_shift_range=Config.WIDTH_SHIFT,
        height_shift_range=Config.HEIGHT_SHIFT,
        zoom_range=Config.ZOOM_RANGE,
        fill_mode='nearest'
    )

# Visualizar exemplos de augmentation
datagen = create_data_augmentation()

# Pegar uma imagem
sample_img = X_train[0:1]

# Gerar 9 augmentações
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
axes = axes.ravel()

i = 0
for batch in datagen.flow(sample_img, batch_size=1):
    axes[i].imshow(batch[0, :, :, 0], cmap='gray')
    axes[i].set_title(f'Augmentation {i+1}', fontsize=10)
    axes[i].axis('off')
    i += 1
    if i >= 9:
        break

plt.suptitle('Exemplos de Data Augmentation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("✓ Data augmentation configurado")

# 🎓 4. Treinamento do Modelo

## 4.1 Configuração de Treinamento

### Optimizer: Adam

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$
$$\theta_t = \theta_{t-1} - \frac{\alpha}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$

**Parâmetros:**
- $\alpha = 0.001$ (learning rate)
- $\beta_1 = 0.9$ (momentum)
- $\beta_2 = 0.999$ (RMSprop)
- $\epsilon = 10^{-7}$

**Justificativa:** Adam adapta learning rate automaticamente, funciona bem sem tuning extensivo

**Referência:** Kingma & Ba, 2015

### Loss Function: Categorical Crossentropy

$$L = -\sum_{i=1}^{C} y_i \log(\hat{y}_i)$$

onde:
- $C = 10$ (número de classes)
- $y_i$ = rótulo verdadeiro (one-hot)
- $\hat{y}_i$ = predição (softmax)

**Justificativa:** Padrão para classificação multi-classe

## 4.2 Callbacks

### 1. EarlyStopping
- **Monitor:** val_loss
- **Patience:** 15 epochs
- **Restore best weights:** True
- **Justificativa:** Para quando não houver melhora, economiza tempo, previne overfitting

### 2. ReduceLROnPlateau
- **Monitor:** val_loss
- **Factor:** 0.5 (reduz LR pela metade)
- **Patience:** 7 epochs
- **Min LR:** $10^{-7}$
- **Justificativa:** Ajusta LR quando estacionar, permite convergência mais fina

### 3. ModelCheckpoint
- **Monitor:** val_accuracy
- **Save best only:** True
- **Justificativa:** Garante que salvamos o melhor modelo

In [ ]:
# Compilar modelo
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=Config.LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("✓ Modelo compilado")
print(f"  Optimizer: Adam (lr={Config.LEARNING_RATE})")
print(f"  Loss: Categorical Crossentropy")
print(f"  Metrics: Accuracy")

In [ ]:
# Configurar callbacks
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-7,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        'best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print("✓ Callbacks configurados:")
print("  - EarlyStopping (patience=15)")
print("  - ReduceLROnPlateau (patience=7)")
print("  - ModelCheckpoint (best model)")

In [ ]:
print("\n" + "="*70)
print(" INICIANDO TREINAMENTO")
print("="*70)
print(f"Epochs: {Config.EPOCHS}")
print(f"Batch size: {Config.BATCH_SIZE}")
print(f"Data augmentation: Sim")
print(f"\nDataset:")
print(f"  Treino: {len(X_train)} samples")
print(f"  Validação: {len(X_val)} samples")
print(f"\nSteps por epoch: {len(X_train) // Config.BATCH_SIZE}")
print("="*70 + "\n")

# Treinar com data augmentation
datagen = create_data_augmentation()
datagen.fit(X_train)

history = model.fit(
    datagen.flow(X_train, y_train, batch_size=Config.BATCH_SIZE),
    epochs=Config.EPOCHS,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1
)

print("\n✓ Treinamento concluído!")

# 📊 5. Análise do Treinamento

In [ ]:
# Plotar histórico de treinamento
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2, color='steelblue')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2, color='coral')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].legend(loc='lower right', fontsize=10)
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2, color='steelblue')
axes[1].plot(history.history['val_loss'], label='Val Loss', linewidth=2, color='coral')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].legend(loc='upper right', fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Estatísticas finais
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
best_val_acc = max(history.history['val_accuracy'])

print(f"\n📊 Estatísticas de Treinamento:")
print(f"  Train Accuracy (final): {final_train_acc:.4f}")
print(f"  Val Accuracy (final): {final_val_acc:.4f}")
print(f"  Val Accuracy (best): {best_val_acc:.4f}")
print(f"  Overfitting gap: {(final_train_acc - final_val_acc):.4f}")

# 🎯 6. Avaliação no Conjunto de Teste

## Métricas de Avaliação:

### 1. Accuracy (Métrica Principal)
$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

### 2. Precision (por classe)
$$\text{Precision} = \frac{TP}{TP + FP}$$

### 3. Recall (por classe)
$$\text{Recall} = \frac{TP}{TP + FN}$$

### 4. F1-Score (por classe)
$$F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

onde:
- TP = True Positives
- TN = True Negatives
- FP = False Positives
- FN = False Negatives

In [ ]:
print("\n" + "="*70)
print(" AVALIAÇÃO NO CONJUNTO DE TESTE")
print("="*70)

# Avaliar
test_loss, test_accuracy = model.evaluate(X_test, y_test_cat, verbose=0)

print(f"\n🎯 RESULTADO FINAL:")
print(f"  Test Loss: {test_loss:.6f}")
print(f"  Test Accuracy: {test_accuracy:.6f}")
print(f"  Test Accuracy (%): {test_accuracy*100:.4f}%")
print("="*70)

# Predições
y_pred_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)

In [ ]:
# Classification Report
report = classification_report(
    y_test, y_pred,
    target_names=Config.CLASS_NAMES,
    digits=4
)

print("\n" + "="*70)
print(" CLASSIFICATION REPORT")
print("="*70)
print(report)

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(12, 10))

# Normalizar por linha
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

sns.heatmap(
    cm_normalized,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    xticklabels=Config.CLASS_NAMES,
    yticklabels=Config.CLASS_NAMES,
    cbar_kws={'label': 'Proporção'},
    linewidths=0.5,
    linecolor='gray'
)

plt.title('Confusion Matrix (Normalized)', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Accuracy por classe
print("\n" + "="*70)
print(" ACCURACY POR CLASSE")
print("="*70)
for i, class_name in enumerate(Config.CLASS_NAMES):
    class_correct = cm[i, i]
    class_total = cm[i].sum()
    class_acc = class_correct / class_total if class_total > 0 else 0
    print(f"{class_name:15s}: {class_acc:.4f} ({class_correct}/{class_total})")

# 🔍 7. Análise de Predições

In [ ]:
# Visualizar predições corretas e incorretas
correct_indices = np.where(y_pred == y_test)[0]
incorrect_indices = np.where(y_pred != y_test)[0]

print(f"Predições corretas: {len(correct_indices)} ({len(correct_indices)/len(y_test)*100:.2f}%)")
print(f"Predições incorretas: {len(incorrect_indices)} ({len(incorrect_indices)/len(y_test)*100:.2f}%)")

# Mostrar exemplos corretos
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.ravel()

indices = np.random.choice(correct_indices, 10, replace=False)

for i, idx in enumerate(indices):
    img = X_test[idx]
    true_label = y_test[idx]
    pred_label = y_pred[idx]
    confidence = y_pred_prob[idx][pred_label]
    
    axes[i].imshow(img.reshape(28, 28), cmap='gray')
    axes[i].axis('off')
    title = f'{Config.CLASS_NAMES[true_label]}\nConf: {confidence:.2f}'
    axes[i].set_title(title, fontsize=9, color='green')

plt.suptitle('✓ Predições CORRETAS (Exemplos)', fontsize=14, fontweight='bold', color='green')
plt.tight_layout()
plt.show()

In [ ]:
# Mostrar exemplos incorretos
if len(incorrect_indices) > 0:
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    axes = axes.ravel()
    
    num_show = min(10, len(incorrect_indices))
    indices = np.random.choice(incorrect_indices, num_show, replace=False)
    
    for i, idx in enumerate(indices):
        img = X_test[idx]
        true_label = y_test[idx]
        pred_label = y_pred[idx]
        confidence = y_pred_prob[idx][pred_label]
        
        axes[i].imshow(img.reshape(28, 28), cmap='gray')
        axes[i].axis('off')
        title = f'True: {Config.CLASS_NAMES[true_label]}\n'
        title += f'Pred: {Config.CLASS_NAMES[pred_label]}\n'
        title += f'Conf: {confidence:.2f}'
        axes[i].set_title(title, fontsize=8, color='red')
    
    # Esconder axes vazias
    for i in range(num_show, 10):
        axes[i].axis('off')
    
    plt.suptitle('✗ Predições INCORRETAS (Exemplos)', fontsize=14, fontweight='bold', color='red')
    plt.tight_layout()
    plt.show()
else:
    print("Nenhuma predição incorreta! Modelo perfeito! 🎉")

# 💾 8. Salvar Resultados

In [ ]:
# Salvar modelo
model.save('dualpath_attentionnet_final.h5')
print("✓ Modelo salvo: dualpath_attentionnet_final.h5")

# Salvar resultados em JSON
results = {
    'model_name': 'DualPath-AttentionNet',
    'test_accuracy': float(test_accuracy),
    'test_loss': float(test_loss),
    'test_accuracy_percent': float(test_accuracy * 100),
    'total_params': int(model.count_params()),
    'training_epochs': len(history.history['loss']),
    'best_val_accuracy': float(max(history.history['val_accuracy'])),
    'configuration': {
        'batch_size': Config.BATCH_SIZE,
        'learning_rate': Config.LEARNING_RATE,
        'l2_reg': Config.L2_REG,
        'dropout_rate': Config.DROPOUT_RATE,
        'spatial_dropout': Config.SPATIAL_DROPOUT,
        'data_augmentation': True
    },
    'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

with open('results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("✓ Resultados salvos: results.json")

# Mostrar resumo final
print("\n" + "="*70)
print(" 🎉 TREINAMENTO CONCLUÍDO COM SUCESSO! 🎉")
print("="*70)
print(f"\n📊 RESULTADO FINAL:")
print(f"\n  🎯 Test Accuracy: {test_accuracy:.6f}")
print(f"  🎯 Test Accuracy: {test_accuracy*100:.4f}%")
print(f"\n  📈 Best Val Accuracy: {max(history.history['val_accuracy']):.6f}")
print(f"  📉 Test Loss: {test_loss:.6f}")
print(f"  🔢 Total Parâmetros: {model.count_params():,}")
print(f"  ⏱️  Epochs Treinados: {len(history.history['loss'])}")
print("\n" + "="*70)
print(" Arquivos salvos:")
print("   - dualpath_attentionnet_final.h5 (modelo completo)")
print("   - best_model.h5 (melhor modelo)")
print("   - results.json (resultados detalhados)")
print("="*70)

# 📝 9. Conclusões e Discussão

## 9.1 Resumo da Arquitetura

Criamos a **DualPath-AttentionNet**, uma arquitetura CNN completamente inédita que combina:

1. **Processamento Dual-Path:** Dois caminhos paralelos processam a imagem em diferentes escalas
2. **Atenção Espacial:** Mecanismo customizado que foca em regiões importantes
3. **Convoluções Dilatadas:** Aumentam receptive field sem aumentar parâmetros
4. **Fusão Adaptativa:** Combina informações das diferentes escalas
5. **Regularização Avançada:** BN + Spatial Dropout + L2

## 9.2 Diferencial das Arquiteturas Clássicas

Nossa arquitetura NÃO replica:
- ❌ LeNet: Muito simples, sem dual-path
- ❌ AlexNet: Processamento sequencial, sem atenção
- ❌ VGG: Apenas 3x3 sequenciais, sem paralelismo
- ❌ ResNet: Residual connections vs dual-path
- ❌ InceptionNet: Inception modules diferentes de nosso dual-path

## 9.3 Pontos Fortes

✅ **Multi-escala:** Captura tanto detalhes quanto contexto

✅ **Atenção:** Foca automaticamente em regiões importantes

✅ **Eficiência:** Convoluções dilatadas aumentam RF sem custos

✅ **Regularização:** Múltiplas técnicas previnem overfitting

✅ **Otimização:** Data augmentation + callbacks inteligentes

## 9.4 Desafios do Fashion-MNIST

O Fashion-MNIST apresenta desafios únicos:

1. **Classes similares:**
   - T-shirt vs Shirt
   - Sneaker vs Ankle boot
   - Pullover vs Coat

2. **Variações intra-classe:**
   - Diferentes estilos de roupas
   - Diferentes poses

3. **Imagens pequenas:**
   - 28x28 pixels
   - Necessidade de preservar detalhes

Nossa arquitetura dual-path com atenção aborda esses desafios!

## 9.5 Referências Principais

1. **Ioffe & Szegedy, 2015** - Batch Normalization
2. **Kingma & Ba, 2015** - Adam Optimizer
3. **Woo et al., 2018** - CBAM (Attention)
4. **Yu & Koltun, 2016** - Dilated Convolutions
5. **Lin et al., 2014** - Network in Network (1x1 conv)
6. **Srivastava et al., 2014** - Dropout
7. **Perez & Wang, 2017** - Data Augmentation

## 9.6 Possíveis Melhorias Futuras

- 🔄 Testar diferentes taxas de dilatação
- 🔄 Adicionar mais blocos dual-path
- 🔄 Experimentar com channel attention
- 🔄 Ensemble com outros modelos
- 🔄 Test-time augmentation
- 🔄 Hyperparameter tuning mais extensivo

---

# ✨ FIM DO NOTEBOOK

**Desenvolvido para o DESAFIO 03e - AC3**

**Arquitetura:** DualPath-AttentionNet (100% inédita)

**Dataset:** Fashion-MNIST

**Documentação:** Completa com justificativas teóricas e cálculos

---